# Bài 22: So khớp ảnh nổi

Hai camera cùng nhìn một cảnh từ hai vị trí hơi khác nhau sẽ thấy cùng các điểm 3D bị dịch chuyển với lượng khác nhau tùy theo độ sâu &mdash; điểm gần dịch nhiều hơn, điểm xa dịch ít hơn. **So khớp ảnh nổi (stereo matching)** tìm các độ dịch này (gọi là **độ chênh lệch — disparity**) tại mọi điểm ảnh, rồi chuyển chúng thành độ sâu thông qua phép tam giác đạc (triangulation). Vấn đề then chốt là sự tương ứng (correspondence): tìm, cho mỗi điểm ảnh trong ảnh trái, điểm ảnh khớp với nó trong ảnh phải. Bài này xây dựng một **bộ so khớp khối (block matcher)** giải quyết bài toán bằng cách so sánh các mảng nhỏ với tổng bình phương sai khác (sum-of-squared differences — SSD) &mdash; một họ hàng gần của luồng quang học ở Bài 21, nhưng bị giới hạn ở tìm kiếm 1D dọc theo một hàng duy nhất thay vì cả một vùng lân cận 2D.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['tsukuba_left.png', 'tsukuba_right.png']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# shared colormap: invalid/masked-out disparities (NaN) render as gray instead of viridis's default
invalid_cmap = plt.cm.viridis.copy()
invalid_cmap.set_bad('gray')

## Ảnh nổi đã nắn thẳng: tại sao việc tìm kiếm là 1D

Với một cặp camera đặt cạnh nhau, hướng về cùng một phía, với các mặt phẳng ảnh song song (một cặp ảnh nổi **đã nắn thẳng (rectified)** &mdash; các hệ camera thực được hiệu chỉnh và biến dạng để xấp xỉ điều này), một sự thật cơ bản của hình học epipolar được áp dụng: điểm tương ứng của bất kỳ điểm ảnh nào trong ảnh trái đều nằm **trên cùng một hàng** trong ảnh phải. Điều này thu gọn việc tìm điểm khớp từ tìm kiếm trên ảnh 2D xuống quét 1D dọc theo một hàng, và độ lệch ngang giữa hai vị trí khớp nhau chính là **độ chênh lệch** $d$.

Độ chênh lệch liên hệ với độ sâu theo công thức

$$Z = \frac{f \cdot B}{d}$$

trong đó $f$ là tiêu cự và $B$ là đường cơ sở (baseline — khoảng cách giữa hai tâm camera). Với một cặp đã nắn thẳng, độ sâu **tỉ lệ nghịch với độ chênh lệch**: vật ở gần có độ chênh lệch lớn, vật ở xa có độ chênh lệch nhỏ, và một vật ở xa vô cùng có độ chênh lệch bằng 0.

## Một cặp ảnh nổi tổng hợp với dữ liệu chuẩn đã biết

Ta tạo một ảnh trái gồm kết cấu ngẫu nhiên thuần túy (để mọi mảng đều đặc biệt về mặt cục bộ &mdash; không có sự mơ hồ của bài toán khẩu độ) và dựng ảnh phải bằng cách dịch mỗi điểm ảnh sang trái theo độ chênh lệch thực của nó, mà ta đặt bằng ba giá trị hằng số khác nhau cho ba "mặt phẳng" độ sâu: một nền và hai hình chữ nhật ở gần hơn. Các bề mặt gần dịch nhiều hơn các bề mặt xa, nên chúng làm lộ ra các điểm ảnh nền ở một bên mà hoàn toàn không có điểm ảnh tương ứng nào trong ảnh trái &mdash; một hiện tượng **che khuất (occlusion)**, được ta vá bằng nhiễu ngẫu nhiên mới để mọi điểm ảnh của ảnh phải vẫn có giá trị nào đó.

In [ ]:
rng = np.random.default_rng(0)
h, w = 150, 200
left = rng.integers(0, 255, (h, w)).astype(np.uint8)

true_disparity = np.full((h, w), 5, dtype=np.int32)     # background         (disparity =  5)
true_disparity[30:120, 50:150] = 15                     # a close rectangle  (disparity = 15)
true_disparity[60:90, 80:120] = 25                      # a closer rectangle (disparity = 25)

right = np.full((h, w), -1, dtype=np.int32)
for y in range(h):
    for x in range(w):
        xr = x - true_disparity[y, x]
        if 0 <= xr < w:
            right[y, xr] = left[y, x]
occluded = right == -1   # positions no left pixel maps to: disocclusions, filled with fresh noise
right[occluded] = rng.integers(0, 255, occluded.sum())
right = right.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(left, cmap='gray')
axes[0].set_title('Left image', fontsize=9)
axes[1].imshow(right, cmap='gray')
axes[1].set_title('Right image', fontsize=9)
im = axes[2].imshow(true_disparity, cmap='viridis')
axes[2].set_title('Ground-truth disparity', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

## So khớp khối dọc theo đường quét

Với mỗi điểm ảnh trong ảnh trái, ta trượt một cửa sổ nhỏ dọc theo *cùng hàng* của ảnh phải trên một khoảng các độ chênh lệch ứng viên và giữ lại độ chênh lệch có tổng bình phương sai khác nhỏ nhất. Phần cài đặt bên dưới có ba vòng lặp Python lồng nhau, nên nó chậm &mdash; ta chạy nó trên một vùng cắt nhỏ thay vì toàn bộ ảnh.

In [ ]:
def block_match_stereo(left, right, block_size=7, max_disp=30):
    h, w = left.shape
    half = block_size // 2
    left_f, right_f = left.astype(np.float64), right.astype(np.float64)
    disparity_map = np.zeros((h, w), dtype=np.float64)

    for y in range(half, h - half):
        for x in range(half, w - half):
            left_patch = left_f[y - half:y + half + 1, x - half:x + half + 1]
            best_d, best_cost = 0, np.inf
            for d in range(max_disp + 1):
                xr = x - d
                if xr - half < 0:
                    break
                right_patch = right_f[y - half:y + half + 1, xr - half:xr + half + 1]
                cost = ((left_patch - right_patch)**2).sum()
                if cost < best_cost:
                    best_cost, best_d = cost, d
            disparity_map[y, x] = best_d
    return disparity_map

# a small crop, since the pure-Python pixel loop is slow
crop = np.s_[0:100, 0:100]
manual_disp = block_match_stereo(left[crop], right[crop], block_size=7, max_disp=30)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(left[crop], cmap='gray')
axes[0].set_title(f'Left crop ({left[crop].shape[0]}x{left[crop].shape[1]})', fontsize=9)
axes[1].imshow(true_disparity[crop], cmap='viridis', vmin=0, vmax=30)
axes[1].set_title(f'True disparity ({true_disparity[crop].shape[0]}x{true_disparity[crop].shape[1]})', fontsize=9)
im = axes[2].imshow(manual_disp, cmap='viridis', vmin=0, vmax=30)
axes[2].set_title(f'Recovered disparity ({manual_disp.shape[0]}x{manual_disp.shape[1]})', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

Ảnh động này cho thấy quá trình tìm kiếm cho một điểm ảnh: cửa sổ đứng yên trong vùng cắt trái trong khi nó trượt dọc theo cùng hàng của vùng cắt phải, mỗi lần một độ chênh lệch ứng viên, với chi phí tổng bình phương sai khác (SSD) được vẽ trực tiếp. Cực tiểu của đường cong chi phí (trong trường hợp này) là độ chênh lệch thực, mà điểm đánh dấu tốt-nhất-cho-đến-nay khóa chặt vào.

<img src="../animations/block_matching_demo.gif" alt="Ảnh động so khớp khối cho một điểm ảnh: một cửa sổ tìm kiếm trượt dọc theo đường quét trong ảnh phải trong khi đường cong chi phí SSD theo độ chênh lệch được dựng trực tiếp và khóa vào cực tiểu tại độ chênh lệch thực" style="display: block; margin: 0 auto; max-width: 720px;">

Có thể bỏ qua dải viền dọc theo các cạnh của vùng cắt: cửa sổ trượt không còn đủ chỗ để tìm kiếm ở đó, một hiệu ứng biên rất giống cách xử lý biên của tích chập (Bài 10). Ra khỏi dải viền đó, và bên trong các hình chữ nhật, độ chênh lệch khôi phục được khớp với dữ liệu chuẩn gần như hoàn hảo, tách sạch cả ba mặt phẳng độ sâu. Ngoại lệ là dải lấm tấm ở cạnh mỗi hình chữ nhật: một cửa sổ nằm vắt qua ranh giới độ sâu trộn lẫn các điểm ảnh từ hai độ chênh lệch thực khác nhau, nên không có độ chênh lệch ứng viên đơn lẻ nào khớp tốt với cả cửa sổ, và việc so khớp trở nên không đáng tin cậy &mdash; một lời báo trước cho bài toán che khuất sẽ được giải quyết tiếp theo. Ảnh chụp thật còn khắt khe hơn trường hợp đồ chơi này: kết cấu lặp lại, vùng phẳng, điểm phản xạ gương (specular highlights) và đổ bóng đều bổ sung thêm sự mơ hồ thực sự bên cạnh che khuất, đó là lý do so khớp ảnh nổi khó trong thực tế.

## Bắt các cặp khớp sai: kiểm tra nhất quán trái-phải

Che khuất và sự mơ hồ ở ranh giới độ sâu tạo ra các cặp khớp sai, như các đốm lấm tấm ở trên. Cách khắc phục: so khớp cả từ trái sang phải và từ phải sang trái, rồi chỉ giữ lại các điểm ảnh mà hai kết quả đồng ý với nhau, loại bỏ phần còn lại là không đáng tin cậy. Tính độ chênh lệch từ phải sang trái không cần code mới &mdash; lật cả hai ảnh theo chiều trái-phải, hoán đổi ảnh nào đóng vai "trái", chạy đúng bộ so khớp đó, rồi lật kết quả trở lại. Một điểm ảnh không vượt qua phép kiểm tra bất cứ khi nào hai chiều lệch nhau quá khoảng một điểm ảnh. Điều này đặc biệt hiệu quả trong việc bắt các vùng bị che khuất: một điểm ảnh bị che sau một bề mặt gần hơn trong góc nhìn kia hoàn toàn không có điểm khớp thực sự ở đó.

In [ ]:
block_size, max_disp = 7, 30
crop_lr = np.s_[0:100, 0:100]
left_c, right_c = left[crop_lr], right[crop_lr]

disp_lr = block_match_stereo(left_c, right_c, block_size=block_size, max_disp=max_disp)
disp_rl = block_match_stereo(right_c[:, ::-1], left_c[:, ::-1], block_size=block_size, max_disp=max_disp)[:, ::-1]

h_c, w_c = disp_lr.shape
consistent = np.zeros((h_c, w_c), dtype=bool)
for y in range(h_c):
    for x in range(w_c):
        d = int(disp_lr[y, x])
        xr = x - d
        if 0 <= xr < w_c:
            consistent[y, x] = abs(disp_rl[y, xr] - d) <= 1

# keep the full crop (no trimming) so the border-truncation effect is visible directly,
# alongside the occlusion-driven inconsistency, in the figure below
disp_checked = np.where(consistent, disp_lr, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, d, title in zip(axes, [disp_lr, disp_rl, disp_checked],
                         ['Left-to-right disparity', 'Right-to-left disparity', 'After consistency check']):
    im = ax.imshow(d, cmap=invalid_cmap, vmin=0, vmax=max_disp)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

print(f'pixels marked inconsistent: {(~consistent).sum()} / {consistent.size}')

Nhiều điểm ảnh đồng ý theo cả hai chiều và vượt qua phép kiểm tra. Những điểm không vượt qua thường do che khuất (các vùng xám lớn cạnh các cạnh dọc của hình chữ nhật) hoặc do kích thước cửa sổ hữu hạn (các vùng xám nhỏ gần các cạnh ngang của hình chữ nhật).

## Cùng việc đó, ở độ phân giải đầy đủ, với OpenCV

`cv2.StereoBM` cài đặt cùng ý tưởng so khớp khối này (với một số tinh chỉnh về hiệu quả và hậu xử lý) đủ nhanh để chạy trên toàn bộ ảnh.

In [ ]:
stereo_bm = cv2.StereoBM_create(numDisparities=32, blockSize=9)
disparity_bm = stereo_bm.compute(left, right).astype(np.float32) / 16.0  # fixed-point output, divide to get pixels

print('recovered disparity vs. ground truth, by region:')
print(f'  background (true=5):  {disparity_bm[100:110, 150:160].mean():.2f}')
print(f'  mid layer (true=15):  {disparity_bm[40:50, 60:70].mean():.2f}')
print(f'  near layer (true=25): {disparity_bm[70:80, 90:110].mean():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
im0 = axes[0].imshow(true_disparity, cmap='viridis', vmin=0, vmax=30)
axes[0].set_title('Ground truth')
im1 = axes[1].imshow(np.where(disparity_bm >= 0, disparity_bm, np.nan), cmap=invalid_cmap, vmin=0, vmax=30)
axes[1].set_title('cv2.StereoBM output\n(gray = invalid)')
for ax in axes:
    ax.axis('off')
plt.colorbar(im1, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

`StereoBM` đánh dấu một điểm ảnh là không hợp lệ (trả về $-1$) ở bất cứ đâu nó không tin chắc vào một cặp khớp duy nhất, ví dụ quá gần biên ảnh để tìm kiếm toàn bộ khoảng độ chênh lệch. Các điểm ảnh này được hiển thị màu xám ở trên.

### Kiểm tra nhất quán tích hợp trong `disp12MaxDiff`

Phép kiểm tra nhất quán trái-phải được tích hợp sẵn trong `cv2.StereoBM` &mdash; đặt `disp12MaxDiff` thành một số dương nhỏ (thường là 1) để bật nó; giá trị mặc định $-1$ để nó tắt.

In [ ]:
stereo_checked = cv2.StereoBM_create(numDisparities=32, blockSize=9)
stereo_checked.setDisp12MaxDiff(1)
disparity_checked = stereo_checked.compute(left, right).astype(np.float32) / 16.0

newly_invalid = (disparity_bm >= 0) & (disparity_checked < 0)
print(f'valid pixels without the check: {(disparity_bm >= 0).sum()}')
print(f'valid pixels with the check:    {(disparity_checked >= 0).sum()}')
print(f'newly marked invalid:           {newly_invalid.sum()}')

plt.imshow(np.where(disparity_checked >= 0, disparity_checked, np.nan), cmap=invalid_cmap, vmin=0, vmax=30)
plt.colorbar(label='disparity (px)')
plt.title('cv2.StereoBM with disp12MaxDiff=1\n(gray = invalid)')
plt.axis('off')
plt.show()

## Kích thước cửa sổ: đánh đổi giữa chi tiết và nhiễu

Một cửa sổ so khớp lớn hơn lấy trung bình trên nhiều điểm ảnh hơn, làm việc so khớp chống chịu nhiễu tốt hơn nhưng làm nhòe qua các điểm gián đoạn độ sâu (trộn các điểm ảnh từ hai độ sâu thực khác nhau thành một ước lượng độ chênh lệch "trung bình" gần các cạnh vật thể). Một cửa sổ nhỏ hơn bảo toàn các ranh giới độ sâu sắc nét nhưng dễ bị đánh lừa bởi nhiễu hoặc kết cấu lặp lại hơn.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, block_size in zip(axes, [5, 15, 31]):
    stereo = cv2.StereoBM_create(numDisparities=32, blockSize=block_size)
    d = stereo.compute(left, right).astype(np.float32) / 16.0
    im = ax.imshow(np.where(d >= 0, d, np.nan), cmap=invalid_cmap, vmin=0, vmax=30)
    ax.set_title(f'blockSize={block_size}', fontsize=9)
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

## Một cặp ảnh nổi thật: Tsukuba

Mọi thứ từ đầu đến giờ đều dùng ảnh tổng hợp, chính là để biết được dữ liệu chuẩn. Với ảnh chụp thật, cùng ba lượt đó (trái-sang-phải, phải-sang-trái, và phép kiểm tra nhất quán giữa chúng) được áp dụng y như trước, lần này trên một cặp ảnh nổi đã nắn thẳng thực sự.

Nhấp nháy giữa hai khung hình giúp dễ dàng nhìn thấy trực tiếp độ dịch &mdash; hãy quan sát chiếc đèn di chuyển xa hơn bao nhiêu so với các giá sách phía sau nó, chính là độ dịch phụ thuộc độ sâu mà bài này đã ước lượng suốt từ đầu.

<img src="../img/tsukuba_flicker.gif" alt="Cặp ảnh nổi Tsukuba trái/phải nhấp nháy" style="display: block; margin: 0 auto; max-width: 400px;">

<p class="image-attribution">Nguồn ảnh: <a href="https://vision.middlebury.edu/stereo/data/scenes2001/" target="_blank" rel="noopener">Middlebury Stereo Datasets (University of Tsukuba)</a></p>

In [ ]:
tsukuba_left = cv2.imread('../img/tsukuba_left.png', cv2.IMREAD_GRAYSCALE)
tsukuba_right = cv2.imread('../img/tsukuba_right.png', cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(tsukuba_left, cmap='gray')
axes[0].set_title('Left photo', fontsize=9)
axes[1].imshow(tsukuba_right, cmap='gray')
axes[1].set_title('Right photo', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
num_disp = 64  # must be divisible by 16; large enough that the nearest surfaces don't clip the search range
stereo = cv2.StereoBM_create(numDisparities=num_disp, blockSize=15)
disp_lr = stereo.compute(tsukuba_left, tsukuba_right).astype(np.float32) / 16.0
disp_rl = stereo.compute(tsukuba_right[:, ::-1], tsukuba_left[:, ::-1]).astype(np.float32) / 16.0
disp_rl = disp_rl[:, ::-1]

stereo_checked = cv2.StereoBM_create(numDisparities=num_disp, blockSize=15)
stereo_checked.setDisp12MaxDiff(1)
disp_checked = stereo_checked.compute(tsukuba_left, tsukuba_right).astype(np.float32) / 16.0

changed = disp_lr != disp_checked
print(f'pixels changed by the consistency check: {changed.sum()} / {changed.size} ({100 * changed.sum() / changed.size:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, d, title in zip(axes, [disp_lr, disp_rl, disp_checked],
                         ['Left-to-right disparity', 'Right-to-left disparity', 'After consistency check']):
    im = ax.imshow(np.where(d >= 0, d, np.nan), cmap=invalid_cmap, vmin=0, vmax=num_disp)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.colorbar(im, ax=axes, orientation='vertical', fraction=0.046, pad=0.04, label='disparity (px)')
plt.show()

Chiếc đèn, vật gần nhất trong cảnh, nổi bật với độ chênh lệch lớn nhất; các giá sách lùi dần một cách mượt mà phía sau nó. Chỉ khoảng 1% điểm ảnh bị đánh dấu, nên thoạt nhìn khung thứ ba trông gần giống khung thứ nhất &mdash; nhưng các đốm lấm tấm đó tập trung chính xác dọc theo các điểm gián đoạn độ sâu khắp cảnh: gáy sách, cạnh giá, hình bóng của cái đầu, bất cứ nơi nào một cửa sổ nằm vắt qua hai độ sâu khác nhau hoặc một camera nhìn thấy thứ mà camera kia không thấy.

## Từ độ chênh lệch đến bản đồ độ sâu

Cho một tiêu cự và một đường cơ sở (theo bất kỳ đơn vị nhất quán nào), $Z = fB/d$ chuyển trực tiếp một bản đồ độ chênh lệch thành một bản đồ độ sâu theo đơn vị đo thực (metric).

In [ ]:
focal_length_px = 500.0
baseline_m = 0.1

valid = disparity_bm > 0
depth_m = np.full_like(disparity_bm, np.nan)
depth_m[valid] = focal_length_px * baseline_m / disparity_bm[valid]

plt.imshow(depth_m, cmap='viridis_r')
plt.colorbar(label='estimated depth (m)')
plt.title('Depth map (nearer = brighter)')
plt.axis('off')
plt.show()

In [ ]:
step = 3  # subsample for a readable, fast-to-render point cloud
ys, xs = np.mgrid[0:depth_m.shape[0]:step, 0:depth_m.shape[1]:step]
zs = depth_m[::step, ::step]
valid_pts = ~np.isnan(zs)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection='3d')
sc = ax.scatter(xs[valid_pts], ys[valid_pts], zs[valid_pts], c=zs[valid_pts], cmap='viridis_r', s=3)
ax.set_xlabel('x (px)')
ax.set_ylabel('y (px)')
ax.set_zlabel('depth (m)')
ax.invert_yaxis()  # match image row convention (row 0 at top)
ax.invert_zaxis()  # nearer (smaller depth) points appear higher, like the disparity peaks above
ax.view_init(elev=20, azim=-60)
#fig.colorbar(sc, ax=ax, shrink=0.6, label='depth (m)')
ax.set_title('The depth map, lifted into 3D')
plt.tight_layout()
plt.show()

Cùng những con số đó, khi nhìn dưới dạng các điểm lơ lửng trong không gian thay vì màu sắc trên một ảnh phẳng, làm cho các "mặt phẳng" trong cảnh tổng hợp này trở thành hiện thực: ba bậc thềm phẳng ở ba độ cao khác nhau, với các lỗ hổng ở nơi `StereoBM` không có cặp khớp đáng tin bị cắt gọn ra khỏi mỗi bậc.

### Bài tập

1. Thay ảnh `left` có kết cấu ngẫu nhiên bằng một vùng xám phẳng cho phần nền (giữ hai hình chữ nhật có kết cấu). Chạy lại `cv2.StereoBM` và mô tả điều gì xảy ra với ước lượng độ chênh lệch trên vùng phẳng &mdash; đây là bài toán khẩu độ ở Bài 21, giờ trong bối cảnh so khớp ảnh nổi.
2. Thử `cv2.StereoSGBM_create` (so khớp khối bán toàn cục (semi-global block matching), áp đặt tính trơn giữa các độ chênh lệch lân cận thay vì so khớp mỗi điểm ảnh hoàn toàn độc lập) thay cho `StereoBM`. So sánh lượng nhiễu lấm tấm ở các vùng phẳng/nền giữa hai phương pháp.
3. Tăng `baseline_m` trong phép chuyển đổi độ sâu. Bản đồ độ sâu ước lượng thay đổi thế nào, và tại sao một hệ ảnh nổi có đường cơ sở rộng hơn lại cho ước lượng độ sâu chính xác hơn với các vật ở xa (với cái giá là một điểm mù khoảng cách tối thiểu lớn hơn gần camera, do khoảng độ chênh lệch và che khuất)?